# ☁️ Dia 79 — Auditando IAM na nuvem: quem tem o quê

**Investigador Cyber 90 Dias** · Fase 6 · Rank Jutsu `A`

> Baseado no *Day 79* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-6-cloud-ai-system-investigation/day-79.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A Atende+ mandou um PDF dizendo que o desenvolvedor externo tem "somente leitura de logs". Rafa nem abre: *"Me traz o que a nuvem diz que deu."*

Neste notebook você vai:

1. **Achatar as políticas** do GCP (projeto, conta de serviço, bucket).
2. **Rodar o `inventory.py` do original**, comentado, e gerar o inventário de 38 concessões.
3. **Conferir as contagens** (e descobrir por que o original diz 17).
4. **AWS:** ver a consulta de curinga perder `vendor-sam`.
5. **Quiz direta/herdada/indireta** e o inventário da Lia.

🐍 Só biblioteca padrão do Python. Os dados do caso Blue Harbor vêm do GitHub do original; sem rede, o notebook usa a cópia embutida (com o mesmo SHA-256). Os comandos `jq` do README viram Python equivalente aqui. Rode as células de cima para baixo (Shift+Enter).

## 1. Evidência com custódia

O `inventory.py` do original lê arquivos de `resources/case-blueharbor/`. A célula abaixo carrega a evidência e **grava** essa pasta, para o script rodar exatamente como no README.

In [ ]:
# ------------------------------------------------------------
# CARREGADOR DE EVIDÊNCIAS (offline-first, com custódia)
# 1. Tenta baixar cada arquivo do repositório original no GitHub.
# 2. Confere o SHA-256 contra o valor esperado (o hash do upstream).
# 3. Se a rede falhar OU o hash não bater, usa a cópia embutida aqui.
# ------------------------------------------------------------
import base64          # a cópia embutida está em base64 (texto puro)
import gzip            # ...e comprimida com gzip, para o notebook não ficar enorme
import hashlib         # SHA-256: a impressão digital de cada arquivo
import json            # os logs de nuvem são JSON
import urllib.request  # baixar da internet sem instalar nada

BASE_RAW = "https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-6-cloud-ai-system-investigation/resources/"

# Hash esperado de cada arquivo (calculado na cópia do upstream deste repositório).
HASH_ESPERADO = {
    "case-blueharbor/gcp/iam-project-policy.json": "ba06f54e053dbe3661bb31490a020aa5142e958566cf499597c10db6c3010ac7",
    "case-blueharbor/gcp/iam-reporting-sa-policy.json": "b6e163a76a2aee374611020535ed8bc878c755ed1ec835b4b9aef378237daaa2",
    "case-blueharbor/gcp/iam-bucket-customer-exports.json": "7f2bd90183b00fc5fb36cc2c195c087c2f8b3bd188fe0f80b28f499bc8359797",
    "case-blueharbor/gcp/groups-export.csv": "3b5b3666f9af782dae68d22e0a5c02fd62982c17c68524cbd5db6a972697c516",
    "case-blueharbor/gcp/service-accounts.csv": "b113a7498497c296e41050e9ff81b13f79893c94a99e241b84bbab43abd7284e",
    "case-blueharbor/aws/iam-authorization-details.json": "ed25c4c9a9b4867e4c499752f639497dd9c1be109d857124c91d2b1b307a1726",
}

# Cópia embutida (plano B offline): o arquivo inteiro, gzip + base64.
_AMOSTRA = {
    "case-blueharbor/gcp/iam-project-policy.json": "H4sIAAAAAAAC/72UWU/CQBDH3/kUTZ/pKuIFiQkoPJigiXeMMWbbDsvKdqfuwRHCd3dbjoBitIT40na7/5n5zezsTEqe51Mbc3OBssuZ9uvei/vneZP8udjtIPsqWBXlQoHsfpyCE/jN1tXl9dttu9nyl4rp/Ou1vPCsQQ14lBtQIe5mKz2zyNS50g+5jLncRJZAEoJaJ1o4bUYRWmnqOupBbAWoQNNGKCz0qApRBVRSMTY80oTThLC5FZ1ZkQgTv/yjTwUpKuOYCvv8VgKFIs8/e+s9J0mtAcKlNlS6gHHCJRlU5iUpF0//sHJaPTnYr9SOjquNSKCN5wJdMOeIBzGkAsc7TRhca6HaPj1t0+woghDNTrkoTwU1XVQJsc66ACBTaNN6TA0NQLIVKAIjmqQuyEqNM991AUBck8pN2l8wQ84+LKgxyeI9chjuGPWv4d8xfChWpp3d0l8QM7P1WAVB8yOKHQfBPnUNsUWZ3Fxkblg43GyAPilu/psAh3KLzsiLr43eIuBgtReXkxwMZZnmfPjcq/ZvaqNO+yy/Dv7AIXCUbrNSmpY+AUAHzyCVBgAA",
    "case-blueharbor/gcp/iam-reporting-sa-policy.json": "H4sIAAAAAAAC/5WQPw/CIBDF934KwtzWP2MTE+vq7GCMw0FPJCnQHLRqjN9dbGvUwUQXyN3j93h314QxLrSttFWeF2wXa8au/RkVg0YgvYS+qci1TVFBgAytWoq6xSOQcJTjGUxTIx/f7tOnD7nYLYbbTzSY3CN1WmIppWttWOOlrIy2A3lLf0jxaVAQNo5CnCLz8BYpAwv1JWjp88evaqRgoHLpDE+/enp5xKqtkf72/HsBm1iOwycjxzGAejCr0/YwhXIm5nLRp+Vd3IZ2Noqz5JbcAXGO8P3EAQAA",
    "case-blueharbor/gcp/iam-bucket-customer-exports.json": "H4sIAAAAAAAC/7WRPW/CMBCG9/wKyzNJ90hIhKpzJQYWxHB2TsYl9kVnB4QQ/x3nAzFAJVq1iy373sf32D5nQkhlfW29CbIUm7QW4jyMqeLQKeR7Ydhsmb5Qx4/aRuJSNR3ugBVxDh6aU7Q6yNlD+vPo8ZvwlN3eIMnUYGo5zOEtpC5gsGjQgD4tO73H8bQRvMxeV15bPP6hxQqh/pFGQD5YjZXW1PlYMrbEMb18HmDxTKqw4AozUTBShSb3qiyp/tZV7az/vWXo2l4zVxT/UXL8mskymyiJEUxPvFf7ucwu2RVMF79crwIAAA==",
    "case-blueharbor/gcp/groups-export.csv": "H4sIAAAAAAAC/53QQQrCMBAF0L1niTlDEbqsQkVclon9tIEkEyapmNsr2I1QW3Q1iz88/swgPMUOnqxTHt5A5tHlEqGEHXY9ZdojDJVxE0YSw6LxIB8dVBRbSAeyspReznWrTtdj3a4imb02FLNNGV+Zpm4OG04irwUFqboj9Cz7OdA39h8IBXIl5bSEOLz2R4StImvGLwelKUaW/FeV92uf3FDPAcQBAAA=",
    "case-blueharbor/gcp/service-accounts.csv": "H4sIAAAAAAAC/5WQsW4CMQyGd57CI0g5dFCpUrdKTF06lAeIfIk5IpL45Dht7+0besDOZkv+v882JQzR+FCmiLPNmMigKrozeatsaiGxCTOOrb/QXOxAJxay+37/avs3u9uvXOg8TZHn9yFWOqMMLF2LxFmDK9uAaTs2zHdwhM5xzbp1nMzhA5ZYMaaH9Q/LJTJ6CJ6yBp3hRJ4ENXDerIQmFg157Ao+p/kM41njDK4W5UQC9HtFAWYPxxdIQYTFLPxOas4k3Q7WrdKQCG5EuCE3pl+V63NqbHPP7nKIXJv1nodFB46zCkdzZdfpf5GB9Tn0cQkClhKKYlYY0F0o+5v0q+bHLXfJY7aJ/wBUinz1CAIAAA==",
    "case-blueharbor/aws/iam-authorization-details.json": "H4sIAAAAAAAC/+1Y23LiOBB9z1e4eJyJB9sQkvjNa5MsO4CJIcnUpPIgbAW8JSxWkpNlp/j3lWxz8RWcpGZ2aoeqpIi71WqdPn3U8bcTSWrcUkgsyICP+j5lDV164E8l6Vv0m9tHgM3500azcbp5JJYMwQKKxx4IgAy8hR+k7T1PWI2eZfxxY/zW/zpuG60b6/bsvHOzczRIILwACXTwQnUfLHRd5R+Nf1r8o4c8UrNoC5NAwKDFf0QATdHasqLJqjpRO7qm6Erro8J/K7sF1wSHy80JH3cZMAbcOfQGfI8Z9EYY+a4P6RaGfShiOITHanN6Q2TFgxLAMDFcF1K63XLPveic4ssysjaLomyDrJNvu5wnYBYl+Bg9WJ/WqtczDDxMZAoWZfUyP5vdu/FNx9S6X4zBqN+tW6+iLXL16sjKmay0J2pLVzS93a6s1xaMTfouDjhaLodri9Rj6jgx7rn1FaW8iyKPIXoaQ/Lsu/AzXBUW08JuuICBCLwfThIhCPVxEJ9Q5XRUZPU8FYI7jRkHIVn/kDJJmXixux8VxoHA64HFFSYTDswUQTrHmPnBLBM+WtN9eoIui+qJEH4pcjFcFmf6kLNxqyjsNWQfClYmVoHsh0bO+liwlQMpDokbgZxbsj49HgIskLNfglxh3vPYMVHjJuQblSNgQQSPcrxdeqmItUE70G55TFN/P54UWfKqUqGEeekpbihxOp4vfgkgSXfOHUDhvvzAv8FiieAnFy8KlO41ukafXZnAJSappshIWyJo4zvT6Y5sZ6LWlbaSXYpuo46sXk6US72t6qr2nW6jBfgHB+PWVYjQ6y+jfJAj7qJKQgSYA1PMBwRnwF1JC58QTKRwiTDwIDmVOMgIuNCTpitpEBnvic8gcTCCEgf4jN/178QcjDwZMnSAM3bf6k76iqLU5kwufo4tLVm5kFvKRNX0iCdvZ8tbL0OerwyfIVmxefaO+SG3YB1pbxRdWwduoVcpZmYOO0lscblqjtTRmsyMtj/kpB0Tdl6PNuy86w4t2xnXIudMRGoe3CtNnnfTKhPh0LsHzJ2L0cYO0OrVilUaqkwftpUSalKzUGLJ5gxZYUp7JUVy7E2RBj3HsZ2aEkJ4pGb5RgalvA3Fs4q2PNiSpe2Yae3qNmyMiB+4/hKgnChw4xXkws53iVABrovDgNFPM4xn2TmgYC7c627KqL479b3P5vdw2vN47j7LTmENEween6zMZTRmPN9Z968QIFpglwrT1EEYnUBV2u0LTb1UWueqenbW0bQLVanWlXWhrmzUZp0iT33Jjq9QeRkyGfMW+Lk0u3Acpy19FDJ7+qcI8+qpmUfR9SkfN+aATKPRU4xvVI7xaiZ/fo8pOiM+OcXcyc7R7xgS10RqhiPDsAa9YW88cYyJ7Rim+aVaag68hDitEEILPoEQsYQ5cQbP+8oWISHoYooe4mZ1a+vR2Ar4P7LcwEgIMydKwlY3wP9lBkmPziWAJ8Cmy5KAe+ygfOx/E3naDYyv9pD73vb7nHXdGrSr2uUX695FRUve49CWjCNxlRFYTD2gv8sLnf8ajQ8Omjkym3371ro3JubvTtew7OHxZD5ir1+UfjulXQHzi4BZtyB1iT+FJRTf8yx/n7nnFL/WLPRCeEYrgkTmXTY/ZyOJ2eRkffIvxIJtXhgaAAA=",
}

EVIDENCIA = {}   # nome do arquivo -> bytes
ORIGEM = {}      # nome do arquivo -> "github" ou "embutida"

for nome in HASH_ESPERADO:
    try:
        # timeout curto: no Colab sem rede (ou num teste offline) não queremos travar
        with urllib.request.urlopen(BASE_RAW + nome, timeout=15) as r:
            dados = r.read()
        if hashlib.sha256(dados).hexdigest() != HASH_ESPERADO[nome]:
            # O arquivo mudou no GitHub: não é a evidência que recebemos. Não usamos.
            raise ValueError("hash diferente do esperado")
        ORIGEM[nome] = "github"
    except Exception as erro:
        dados = gzip.decompress(base64.b64decode(_AMOSTRA[nome]))
        ORIGEM[nome] = f"embutida ({type(erro).__name__})"
    EVIDENCIA[nome] = dados

# Recibo de custódia: confere TODOS os hashes de novo, depois de carregados.
for nome, dados in EVIDENCIA.items():
    ok = hashlib.sha256(dados).hexdigest() == HASH_ESPERADO[nome]
    print(f"{'OK ' if ok else 'FALHA'}  {nome:55s} origem: {ORIGEM[nome]}")


def carregar_json(nome):
    """Devolve o JSON já convertido em objetos Python (list/dict)."""
    return json.loads(EVIDENCIA[nome].decode("utf-8"))


def carregar_jsonl(nome):
    """JSONL = um JSON por linha. Devolve uma lista de dicionários."""
    return [json.loads(l) for l in EVIDENCIA[nome].decode("utf-8").splitlines() if l.strip()]


def carregar_csv(nome):
    """CSV -> lista de dicionários (cabeçalho como chaves)."""
    import csv, io
    return list(csv.DictReader(io.StringIO(EVIDENCIA[nome].decode("utf-8"))))

In [ ]:
from pathlib import Path
for nome, dados in EVIDENCIA.items():
    destino = Path("resources") / nome                 # mesma estrutura do original
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(dados)
print("Pasta resources/ pronta:", sorted(str(p) for p in Path("resources").rglob("*.*")))

## 2. GCP: achatar as políticas

`jq -r '.bindings[] | .role as $r | .members[] | [$r, .] | @tsv'` é um laço duplo: para cada vinculação, para cada membro.

In [ ]:
def tsv(*campos):
    """Imprime separado por tabulação, como o @tsv do jq."""
    print("\t".join(str(c) for c in campos))


def achatar(politica):
    """Uma linha (papel, membro) por par: o 'flatten' do gcloud."""
    for b in politica["bindings"]:                     # cada vinculação tem 1 papel...
        for m in b["members"]:                         # ...e N membros
            yield b["role"], m

proj = carregar_json("case-blueharbor/gcp/iam-project-policy.json")
linhas = list(achatar(proj))
for papel, membro in linhas:
    print(f"{papel:38s} {membro}")
print(f"\n{len(linhas)} linhas (esperado 12). Alguma nomeia sam.reyes?",
      any("sam.reyes" in m for _, m in linhas))

In [ ]:
print("--- Conta de serviço reporting-sa e bucket ---")
for arq in ("case-blueharbor/gcp/iam-reporting-sa-policy.json", "case-blueharbor/gcp/iam-bucket-customer-exports.json"):
    for papel, membro in achatar(carregar_json(arq)):
        print(f"{papel:38s} {membro}")
print("\n--- Inventário de contas de serviço ---")
for sa in carregar_csv("case-blueharbor/gcp/service-accounts.csv"):
    print(sa["email"].split("@")[0], "| anexada a:", sa["attached_to"] or "-", "| chaves antes de 12/set:", sa["user_managed_keys_before_2026_09_12"])

## 3. O `inventory.py` do original, comentado

É o mesmo código do README (só stdlib), com comentários em português. Ele escreve o CSV em `lab-p6/notes/`.

In [ ]:
import csv, json, sys, io

C = "resources/case-blueharbor/"
HOME_DOMAIN = "blueharbor.example"                      # quem NÃO termina com isso é externo
RISKY = {"roles/owner", "roles/editor", "roles/iam.serviceAccountKeyAdmin",
         "roles/iam.serviceAccountTokenCreator", "roles/iam.serviceAccountUser",
         "AdministratorAccess", "AmazonS3FullAccess"}  # papéis amplos ou de escalada

groups = {}                                             # grupo -> lista de usuários
for row in csv.DictReader(open(C + "gcp/groups-export.csv")):
    groups.setdefault("group:" + row["group_email"], []).append("user:" + row["member"])

buf = io.StringIO()                                     # no original é sys.stdout; aqui guardamos
out = csv.writer(buf, lineterminator="\n")
out.writerow(["cloud", "identity", "identity_type", "role_or_policy", "resource", "grant_path", "flag"])

def flag(identity, role):
    f = []
    if role in RISKY:
        f.append("broad-or-escalation-role")
    if identity.startswith("user:") and not identity.endswith("@" + HOME_DOMAIN):
        f.append("external-identity")
    return ";".join(f)

# GCP: cada política, cada vinculação, cada membro; e EXPANDE grupos em usuários
for path, resource in (("gcp/iam-project-policy.json", "project:blueharbor-analytics"),
                       ("gcp/iam-reporting-sa-policy.json", "serviceAccount:reporting-sa"),
                       ("gcp/iam-bucket-customer-exports.json", "bucket:blueharbor-customer-exports")):
    for b in json.load(open(C + path))["bindings"]:
        for m in b["members"]:
            out.writerow(["gcp", m, m.split(":")[0], b["role"], resource, "direct", flag(m, b["role"])])
            for member in groups.get(m, []):           # o grupo vira N linhas "group via"
                out.writerow(["gcp", member, "user", b["role"], resource, "group via " + m,
                              flag(member, b["role"])])

# AWS: políticas gerenciadas, inline (com marcação de curinga e escrita em IAM), de grupo e de roles
aws = json.load(open(C + "aws/iam-authorization-details.json"))
for u in aws["UserDetailList"]:
    for p in u.get("AttachedManagedPolicies", []):
        out.writerow(["aws", u["UserName"], "user", p["PolicyName"], "*", "direct", flag("", p["PolicyName"])])
    for p in u.get("UserPolicyList", []):
        for st in p["PolicyDocument"]["Statement"]:
            acts = st["Action"] if isinstance(st["Action"], list) else [st["Action"]]
            f = ["wildcard-action"] if "*" in acts else []
            if any(a.startswith(("iam:Create", "iam:Put", "iam:Attach")) for a in acts):
                f.append("iam-write")
            out.writerow(["aws", u["UserName"], "user", p["PolicyName"] + " (inline)",
                          st["Resource"], "direct", ";".join(f)])
    for g in u.get("GroupList", []):
        for gd in aws["GroupDetailList"]:
            if gd["GroupName"] == g:
                for p in gd["AttachedManagedPolicies"]:
                    out.writerow(["aws", u["UserName"], "user", p["PolicyName"], "*", "group via " + g, ""])
for r in aws["RoleDetailList"]:
    for p in r.get("RolePolicyList", []):
        for st in p["PolicyDocument"]["Statement"]:
            out.writerow(["aws", r["RoleName"], "role", p["PolicyName"] + " (inline)", st["Resource"], "direct", ""])

Path("lab-p6/notes").mkdir(parents=True, exist_ok=True)
Path("lab-p6/notes/day-79-iam-inventory.csv").write_text(buf.getvalue())
print("CSV gravado em lab-p6/notes/day-79-iam-inventory.csv")

## 4. Conferindo as contagens

O original diz "39 linhas, 17 marcadas". Vamos medir as duas coisas separadamente.

In [ ]:
texto = Path("lab-p6/notes/day-79-iam-inventory.csv").read_text().splitlines()
print("wc -l:", len(texto))                                         # 39
awk_igual = sum(1 for l in texto if l.split(",")[-1] != "")         # o awk -F, '$7 != ""' (conta o cabeçalho!)
INV = list(csv.DictReader(io.StringIO("\n".join(texto))))
marcadas = [r for r in INV if r["flag"]]                            # só concessões
print("awk do original (com cabeçalho):", awk_igual, "| concessões marcadas de verdade:", len(marcadas))
print("\ngrep sam.reyes:")
for r in INV:
    if "sam.reyes" in r["identity"]:
        print(" ", r["role_or_policy"], "|", r["resource"], "|", r["grant_path"], "|", r["flag"])

## 5. AWS: a consulta de curinga e a que realmente acha o problema

In [ ]:
doc = carregar_json("case-blueharbor/aws/iam-authorization-details.json")

print("--- Usuários e políticas gerenciadas ---")
for u in doc["UserDetailList"]:
    nomes = [p["PolicyName"] for p in u.get("AttachedManagedPolicies", [])] or ["no-managed-policy"]
    for n in nomes:
        tsv(u["UserName"], n)

print("\n--- Inline com Action == '*' (consulta de curinga) ---")
for u in doc["UserDetailList"]:
    for p in u.get("UserPolicyList", []):
        for st in p["PolicyDocument"]["Statement"]:
            a = st["Action"]
            if a == "*" or (isinstance(a, list) and "*" in a):
                tsv(u["UserName"], p["PolicyName"])

print("\n--- TODA declaração inline com recurso (a que acha vendor-sam) ---")
for u in doc["UserDetailList"]:
    for p in u.get("UserPolicyList", []):
        for st in p["PolicyDocument"]["Statement"]:
            tsv(u["UserName"], p["PolicyName"], st.get("Sid", "-"), json.dumps(st["Action"]), st["Resource"])

print("\nTag de svc-reporting:", [t["Value"] for u in doc["UserDetailList"] if u["UserName"] == "svc-reporting" for t in u["Tags"]])

## 6. 🎮 Quiz: direta, herdada ou indireta?

Responda **1** (direta), **2** (herdada) ou **3** (indireta).

In [ ]:
MODO_JOGO = True
QUIZ = [
    ("reporting-sa tem storage.objectAdmin na política do bucket", "1",
     "A identidade está nomeada na política do próprio recurso."),
    ("sam.reyes alcança reporting-sa por ser membro de data-eng, que tem serviceAccountKeyAdmin", "3",
     "Passa por um grupo e por uma permissão de emitir credenciais."),
    ("Um papel concedido na pasta 'Analytics' acima do projeto vale para o bucket", "2",
     "Vem de cima, e a Blue Harbor não entregou esse export: não dá para ver."),
    ("Todo viewer do projeto tem legacyBucketReader no bucket (membro projectViewer:)", "3",
     "Valor de conveniência: concede a quem tiver o papel básico no projeto."),
    ("vendor-sam pode criar chave para svc-reporting (RotateOwnKeys com user/*)", "3",
     "Emitir credencial para outro usuário dá o poder dele."),
    ("➕ dev.externo alcança o cadastro_espelho via lia-devs -> key admin em lia-bot -> dataViewer", "3",
     "Mesma forma do Blue Harbor, no caso brasileiro."),
]
acertos = 0
for i, (frase, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {frase}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3"):
            r = input("   1=direta 2=herdada 3=indireta: ").strip()
        acertos += (r == certo); print("   ✅" if r == certo else "   ❌")
    print(f"   Gabarito: {dict(zip('123', ['direta','herdada','indireta']))[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 7. ➕ Operação Isca: o inventário da Lia

Mesmo script, com `HOME_DOMAIN = "serraazul.example"` e uma marcação nova: `dado-pessoal` quando o recurso tem coluna `cpf`.

In [ ]:
# ------------------------------------------------------------
# DADOS DA OPERAÇÃO ISCA (fictícios) — mesmo esquema offline-first
# ------------------------------------------------------------
import base64, gzip, hashlib, json, urllib.request

BASE_ISCA = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/"
_ISCA = {
    "dia-79/lia-iam.json": "H4sIAAAAAAAC/71Vy24TMRTd5yusWYHUDKELFl01fSBVUJDKY4Mq6459M3Xw+A62J22o+B5WLPmC/hi2ZxIlLUmZtmI1flyfc+5zrgcs444Ki9key46varIebn7e/EbHJLKT8SmTxGpLU/TEHFoL8L3RQ62APZso4W9+CUXPc/aabAXBRBlXKwvhEVasRD9UUA1r0krM82wnsHVYge56wFgGjVT+kMxElS6cfQln6SJcBbaZEkkYaP2h3bkIwhYP31J56+3idbDQVH6c1+n9+Oj05B0/Ox4fZe3tj/Q5H3TL8wiaFcpIZe7osKQTSPy6F3Rp0C5EVFgVaFfIs9JSU+95NRRE9f4yYDleQVUHnBXenY0ctjE5yEqZzTxdcMZCUGP8nsRa03wo1P5ajvIQ/bzsbKG17SdlpvDyfn8D1VDizO2DRyOxAuX60RSq/NagnedTKj65bYS3HI/MBfmHuT1IiygrE2Q8cIm8fUU8wPAAvKjTf6qNSLuu7w3Ox9sT+UQBvEvdK4yPrJ+VQErw4NBzARKct8TR1agvqFcgl+UQ0d7/h45bI/x8T80/dQmm4SNINwaWNJmSPDqEcZa2UjJRT7qVoQq7pUeNEzKLbSwd3a2hRCMUdDunfAMCKFtJVtGIryFXsdZ9VXNM89/1SpXzZANRTsUUhR/3mlqPitzOvZI0liDmB8nHMwS5LafptyR8m/q9NUEbCr20TU0LiKSjPYo6trVzm42owCbbYBfuPFpDG7o/ufoIBjCQaxXc74G/uZ9uo08JKPfq750XAna+HK9uZb6uxa0t2s6rfgXRVboJewkcIsihpkays8awCJdcVhUav2yjC5ih4yXa1B5hTHEwHpO63dHuK/5yxEe7AWm0HpulyocMy406Dxql5YOUhdAOfgz+AI1eFJ3CCQAA",
}
ISCA = {}
for nome, b64 in _ISCA.items():
    embutido = gzip.decompress(base64.b64decode(b64))            # a cópia que veio com o notebook
    try:
        with urllib.request.urlopen(BASE_ISCA + nome, timeout=15) as r:
            baixado = r.read()
        # só aceita o arquivo da rede se for idêntico ao embutido (mesmo SHA-256)
        ISCA[nome] = baixado if hashlib.sha256(baixado).digest() == hashlib.sha256(embutido).digest() else embutido
        origem = "github" if ISCA[nome] is baixado else "embutida (hash diferente)"
    except Exception as erro:
        ISCA[nome], origem = embutido, f"embutida ({type(erro).__name__})"
    print(f"{nome:35s} sha256={hashlib.sha256(ISCA[nome]).hexdigest()[:16]}...  origem: {origem}")

In [ ]:
LIAIAM = json.loads(ISCA["dia-79/lia-iam.json"])
HOME_LIA = "serraazul.example"
grupos_lia = {}
for g in LIAIAM["grupos"]:
    grupos_lia.setdefault("group:" + g["grupo"], []).append("user:" + g["membro"])
colunas = set(LIAIAM["dataset_cadastro_espelho"]["colunas"])

def flag_lia(identidade, papel, recurso):
    f = []
    if papel in RISKY:
        f.append("broad-or-escalation-role")
    if identidade.startswith("user:") and not identidade.endswith("@" + HOME_LIA):
        f.append("external-identity")
    if recurso.startswith("dataset:") and "cpf" in colunas and "bigquery.data" in papel:
        f.append("dado-pessoal")                       # leitura de dataset com CPF
    return ";".join(f)

INV_LIA = []
for chave, recurso in (("projeto", "project:serraazul-lia"),
                       ("conta_de_servico_lia_bot", "serviceAccount:lia-bot"),
                       ("dataset_cadastro_espelho", "dataset:cadastro_espelho"),
                       ("bucket_lia_tmp_exports", "bucket:lia-tmp-exports")):
    for papel, m in achatar(LIAIAM[chave]):
        INV_LIA.append(dict(identity=m, role=papel, resource=recurso, grant_path="direct", flag=flag_lia(m, papel, recurso)))
        for u in grupos_lia.get(m, []):
            INV_LIA.append(dict(identity=u, role=papel, resource=recurso, grant_path="group via " + m, flag=flag_lia(u, papel, recurso)))

for r in INV_LIA:
    print(f"{r['identity'][:42]:42s} {r['role']:34s} {r['resource']:26s} {r['grant_path'][:28]:28s} {r['flag']}")
print("\nChaves de lia-bot antes de 2/out:",
      [s["chaves_gerenciadas_antes_de_2026_10_02"] for s in LIAIAM["contas_de_servico"] if s["email"].startswith("lia-bot")])

In [ ]:
#@title Notas das linhas marcadas (Lia) { display-mode: "form" }
nota_key_admin = "Atende+ deu ao próprio grupo de devs o poder de criar chaves da lia-bot; nenhuma chave existia antes, então nada dependia disso. Remover. Aprova: Serra Azul (controladora) instrui; Atende+ (operadora) executa."  #@param {type:"string"}
nota_dataviewer = "lia-bot lê o cadastro inteiro (cpf, nome, telefone, email, agencia). Privilégio mínimo: view autorizada só com 'consta/não consta' e agência, sem nome/telefone/email."  #@param {type:"string"}
nota_owner = "ti-coop como owner do projeto: esperado (controladora); revisar membros."  #@param {type:"string"}

linhas = ["# Notas do inventário de IAM — serraazul-lia", "",
          "| Identidade | Papel | Recurso | Caminho | Marcação | Nota |", "|---|---|---|---|---|---|"]
for r in INV_LIA:
    if not r["flag"]:
        continue
    nota = (nota_key_admin if "KeyAdmin" in r["role"] else nota_dataviewer if "dataViewer" in r["role"]
            else nota_owner if r["role"] == "roles/owner" else "(escreva)")
    linhas.append(f"| {r['identity']} | {r['role']} | {r['resource']} | {r['grant_path']} | {r['flag']} | {nota} |")
linhas += ["", "Limite: sem políticas de pasta/organização acima de serraazul-lia; concessões herdadas desconhecidas.",
           "Export a pedir: políticas de IAM da pasta e da organização (via Serra Azul, controladora)."]
Path("notas").mkdir(exist_ok=True)
Path("notas/dia-79-lia-notas.md").write_text("\n".join(linhas) + "\n", encoding="utf-8")
print("\n".join(linhas))
try:
    from google.colab import files
    files.download("notas/dia-79-lia-notas.md")
except ImportError:
    print("\n(Fora do Colab: arquivo em notas/)")

## ✅ Antes de fechar o Dia 79

- [ ] `day-79-iam-inventory.csv` com 38 concessões, notas nas linhas marcadas e coluna `key_permission`
- [ ] Cadeias GCP (Sam → data-eng → reporting-sa → bucket) e AWS (vendor-sam → svc-reporting → mirror) escritas
- [ ] Limite: herança acima do projeto não vista; export a pedir
- [ ] `notas/dia-79-lia-notas.md` com quem aprova cada correção

**Amanhã (Dia 80):** o inventário vira grafo. Qual aresta cortar para quebrar o caminho, inclusive os que o ator não usou?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>